In [7]:
import os
import torch
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.metrics import r2_score, mean_squared_error
from train_lincs_Bio_En import load_biological_mask
from models.PRnet_Bio_Encoder import PRnet

# ==========================================
# 1. 准备环境与加载最佳模型
# ==========================================
# 设置高颜值画图风格 (发 Paper / 汇报 专用)
sns.set_theme(style="ticks", context="talk")
plt.rcParams['font.sans-serif'] = ['Arial']

# 加载数据与 Mask
adata = sc.read('/media/mldadmin/home/s125mdg35_08/PRnet/dataset/Lincs_L1000.h5ad')
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

gmt_path = '/media/mldadmin/home/s125mdg35_08/PRnet baseline/PRnet/dataset/h.all.v2026.1.Hs.symbols.gmt'
mask = load_biological_mask(gmt_path, adata)

# 初始化模型
model = PRnet(
    adata, x_dimension=978, hidden_layer_sizes=[128], z_dimension=64, 
    adaptor_layer_sizes=[128], comb_dimension=64, drug_dimension=1024, 
    comb_num=1, dr_rate=0.05, mask=mask
)

# 加载你刚才确定的最强权重
checkpoint_path = './checkpoint/PRnet_Bio_En/random_split_0_best_epoch_all.pt'
model.PGM.load_state_dict(torch.load(checkpoint_path, map_location='cpu'))
model.eval() # 开启评估模式

# ==========================================
# 2. 抽取测试数据并预测 (这里随机抽 2000 个细胞演示)
# ==========================================
np.random.seed(42)
test_idx = np.random.choice(adata.n_obs, 2000, replace=False)
test_adata = adata[test_idx].copy()

# 构建输入 Tensor
x_test = torch.tensor(test_adata.X.toarray() if hasattr(test_adata.X, 'toarray') else test_adata.X, dtype=torch.float32)
c_test = torch.zeros((x_test.shape[0], 1024), dtype=torch.float32) # 假设基础状态，如果有药物 covariate 请替换
n_test = torch.zeros((x_test.shape[0], 10), dtype=torch.float32)
with torch.no_grad():
    # 此时的 dist 已经是预测好的 Tensor 了，直接转 numpy！
    dist = model.PGM(x_test, c_test, n_test)
    y_pred = dist[:, :978].cpu().detach().numpy()
    y_true = x_test.cpu().numpy()

# 将所有样本的基因展平，计算全局回归指标
y_true_flat = y_true.flatten()
y_pred_flat = y_pred.flatten()

mse_val = mean_squared_error(y_true_flat, y_pred_flat)
r2_val = r2_score(y_true_flat, y_pred_flat)
pcc_val, p_value = pearsonr(y_true_flat, y_pred_flat)

# ==========================================
# 3. 开始画图
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 【图 A】：预测散点图 (展示 PCC 和 R2)
# 为了画图不卡，我们在散点图里随机抽样 10000 个点来画
sample_idx = np.random.choice(len(y_true_flat), 10000, replace=False)
sns.scatterplot(x=y_true_flat[sample_idx], y=y_pred_flat[sample_idx], alpha=0.3, color="#2b8a3e", edgecolor=None, ax=axes[0])

# 画对角线 y=x
min_val, max_val = 0, max(y_true_flat.max(), y_pred_flat.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2.5, label="Perfect Fit (y=x)")

axes[0].set_title(f"Gene Expression Prediction\n$R^2$ = {r2_val:.3f} | PCC = {pcc_val:.3f}", pad=15)
axes[0].set_xlabel("True Expression (log1p)")
axes[0].set_ylabel("Predicted Expression (log1p)")
axes[0].legend()

# 【图 B】：残差分布图 (展示 MSE)
residuals = y_pred_flat - y_true_flat
# 同样抽样画残差分布
sns.histplot(residuals[sample_idx], bins=60, kde=True, color="#1971c2", ax=axes[1])
axes[1].axvline(0, color='r', linestyle='--', lw=2.5)
axes[1].set_title(f"Prediction Residuals\nMSE = {mse_val:.4f}", pad=15)
axes[1].set_xlabel("Residual (Predicted - True)")
axes[1].set_ylabel("Density")

sns.despine()
plt.tight_layout()
plt.savefig("./results/PRnet_Bio_En/Performance_Plots.png", dpi=300)
plt.show()

通路 1: HALLMARK_ADIPOGENESIS -> 匹配到了 25 个基因
通路 2: HALLMARK_ALLOGRAFT_REJECTION -> 匹配到了 24 个基因
通路 3: HALLMARK_ANDROGEN_RESPONSE -> 匹配到了 19 个基因
通路 4: HALLMARK_ANGIOGENESIS -> 匹配到了 7 个基因
通路 5: HALLMARK_APICAL_JUNCTION -> 匹配到了 29 个基因
通路 6: HALLMARK_APICAL_SURFACE -> 匹配到了 6 个基因
通路 7: HALLMARK_APOPTOSIS -> 匹配到了 40 个基因
通路 8: HALLMARK_BILE_ACID_METABOLISM -> 匹配到了 8 个基因
通路 9: HALLMARK_CHOLESTEROL_HOMEOSTASIS -> 匹配到了 17 个基因
通路 10: HALLMARK_COAGULATION -> 匹配到了 12 个基因
通路 11: HALLMARK_COMPLEMENT -> 匹配到了 30 个基因
通路 12: HALLMARK_DNA_REPAIR -> 匹配到了 23 个基因
通路 13: HALLMARK_E2F_TARGETS -> 匹配到了 53 个基因
通路 14: HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION -> 匹配到了 25 个基因
通路 15: HALLMARK_ESTROGEN_RESPONSE_EARLY -> 匹配到了 29 个基因
通路 16: HALLMARK_ESTROGEN_RESPONSE_LATE -> 匹配到了 33 个基因
通路 17: HALLMARK_FATTY_ACID_METABOLISM -> 匹配到了 22 个基因
通路 18: HALLMARK_G2M_CHECKPOINT -> 匹配到了 45 个基因
通路 19: HALLMARK_GLYCOLYSIS -> 匹配到了 41 个基因
通路 20: HALLMARK_HEDGEHOG_SIGNALING -> 匹配到了 5 个基因
通路 21: HALLMARK_HEME_METABOLISM -> 匹配到了 22 个基因
通路

ValueError: Found input variables with inconsistent numbers of samples: [1956000, 3912000]

In [3]:
import os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

# ==========================================
# 第一步：自动扫描定位你的 CSV 文件
# ==========================================
def find_csv_file(keyword):
    # 遍历 results 文件夹及其所有子文件夹
    for root, dirs, files in os.walk('./results/'):
        for file in files:
            if keyword in file and file.endswith('.csv'):
                return os.path.join(root, file)
    raise FileNotFoundError(f"在 ./results/ 及其子目录下找不到包含 '{keyword}' 的 CSV 文件，请检查！")

x_true_path = find_csv_file('x_true_array')
y_true_path = find_csv_file('y_true_array')
y_pred_path = find_csv_file('y_pre_array')
drugs_path = find_csv_file('cov_drug_array')

print("✅ 成功定位到 CSV 文件：")
print(f"x_true 路径 -> {x_true_path}")
print(f"y_true 路径 -> {y_true_path}")
print(f"y_pred 路径 -> {y_pred_path}")
print(f"drugs  路径 -> {drugs_path}\n")

# ==========================================
# 第二步：智能读取 CSV 并转换为 Numpy 矩阵
# ==========================================
def load_csv_safe(path):
    # 读取 csv 文件
    df = pd.read_csv(path)
    # 如果保存时带了索引列 (通常叫 'Unnamed: 0')，我们需要把它剔除，只保留纯数据
    if df.columns[0].startswith('Unnamed'):
        df = df.iloc[:, 1:]
    return df.values

print("正在读取庞大的 CSV 矩阵，这可能需要几秒钟...")
x_true = load_csv_safe(x_true_path)
y_true = load_csv_safe(y_true_path)
y_pred = load_csv_safe(y_pred_path)
drugs = load_csv_safe(drugs_path)

# 确保把一维的药物标签展平为一维数组
if drugs.ndim > 1:
    drugs = drugs.flatten()

print(f"✅ 成功加载 {len(drugs)} 个细胞的测试数据。正在计算 logFC PCC...\n")

# ==========================================
# 第三步：严格对齐 Nature 论文的 logFC PCC 计算
# ==========================================
# 1. 计算对数折叠变化（药物效应向量）
true_logfc = y_true - x_true
pred_logfc = y_pred - x_true

# 2. 按化合物 (drug) 分组并求 978 个基因的平均值
df_true = pd.DataFrame(true_logfc)
df_true['drug'] = drugs
mean_true_logfc_per_drug = df_true.groupby('drug').mean()

df_pred = pd.DataFrame(pred_logfc)
df_pred['drug'] = drugs
mean_pred_logfc_per_drug = df_pred.groupby('drug').mean()

# 3. 遍历计算 PCC
pcc_list = []
unique_drugs = mean_true_logfc_per_drug.index

for drug in unique_drugs:
    vec_true = mean_true_logfc_per_drug.loc[drug].values
    vec_pred = mean_pred_logfc_per_drug.loc[drug].values
    
    # 排除方差为0的异常情况，防止 pearsonr 报错
    if np.std(vec_true) > 0 and np.std(vec_pred) > 0:
        r, _ = pearsonr(vec_true, vec_pred)
        pcc_list.append(r)

final_paper_pcc = np.mean(pcc_list)

print("="*45)
print(f"📊 严格对齐 Nature 论文的最终指标结果:")
print(f"➡️ Pearson of log(FC) in compounds: {final_paper_pcc:.4f}")
print("="*45)

# ==========================================
# 第四步：附赠高颜值橙色 PCC 分布图
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="ticks")

plt.figure(figsize=(9, 6))
sns.histplot(pcc_list, bins=30, kde=True, color="#ff9f43", edgecolor="white", stat="density", alpha=0.7)
plt.axvline(final_paper_pcc, color='red', linestyle='--', linewidth=2.5, label=f'Mean PCC = {final_paper_pcc:.4f}')
plt.title("Pearson Correlation of log(FC) in Compounds", fontsize=14, pad=12)
plt.xlabel("Pearson Correlation Coefficient (PCC)", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.legend()
sns.despine()
plt.tight_layout()

# 将图片保存在自动找到的 x_true 文件所在的同级目录中
save_dir = os.path.dirname(x_true_path)
save_img_path = os.path.join(save_dir, 'Paper_Aligned_PCC_Distribution.png')
plt.savefig(save_img_path, dpi=300)
print(f"✅ 高清分布图已保存至: {save_img_path}")

✅ 成功定位到 CSV 文件：
x_true 路径 -> ./results/lincs/random_split_0_x_true_array.csv
y_true 路径 -> ./results/lincs/random_split_0_y_true_array.csv
y_pred 路径 -> ./results/lincs/random_split_0_y_pre_array.csv
drugs  路径 -> ./results/lincs/random_split_0_cov_drug_array.csv

正在读取庞大的 CSV 矩阵，这可能需要几秒钟...


ParserError: Error tokenizing data. C error: Expected 1 fields in line 2085, saw 2


In [1]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 第一步：全自动地毯式搜索日志文件
# ==========================================
def find_and_parse_logs():
    # 搜索这几个最有可能存放日志的目录
    search_dirs = ['./', './checkpoint/', './results/']
    # 搜索这些后缀的文件
    valid_extensions = ['.txt', '.log', '.out', '.csv']
    
    epochs, mses, r2s = [], [], []
    found_file = None
    
    for d in search_dirs:
        if not os.path.exists(d): continue
        for root, dirs, files in os.walk(d):
            for file in files:
                if any(file.endswith(ext) for ext in valid_extensions):
                    file_path = os.path.join(root, file)
                    try:
                        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                            content = f.read()
                            # 侦测是否包含关键特征字符串
                            if 'Result: MSE=' in content and 'R2=' in content:
                                found_file = file_path
                                f.seek(0) # 回到文件开头准备逐行解析
                                for line in f:
                                    match = re.search(r'Epoch\s+(\d+)\s+Result:\s+MSE=([\d\.]+),\s*R2=([\d\.]+)', line)
                                    if match:
                                        epochs.append(int(match.group(1)) + 1)
                                        mses.append(float(match.group(2)))
                                        r2s.append(float(match.group(3)))
                                if len(epochs) > 0:
                                    return found_file, epochs, mses, r2s
                    except Exception:
                        pass
    return None, [], [], []

print("🕵️ 正在全盘搜索包含 MSE 和 R2 的日志文件，请稍候...")
log_file, epochs, mses, r2s = find_and_parse_logs()

# ==========================================
# 第二步：根据搜索结果决定下一步
# ==========================================
if log_file:
    print(f"✅ 太幸运了！在隐藏文件里找到了日志：{log_file}")
    print(f"📊 成功提取了 {len(epochs)} 轮的数据，正在为您绘制高清曲线图...")
    
    df = pd.DataFrame({'Epoch': epochs, 'MSE': mses, 'R2': r2s})
    
    # 绘图代码
    sns.set_theme(style="ticks", context="talk")
    plt.rcParams['font.sans-serif'] = ['Arial']
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    sns.lineplot(data=df, x='Epoch', y='MSE', color='red', linewidth=2.5, ax=axes[0])
    axes[0].set_title('MSE over Epochs', fontsize=16, pad=15)
    axes[0].set_xlabel('Epoch', fontsize=14)
    axes[0].set_ylabel('Mean Squared Error (MSE)', fontsize=14)
    axes[0].grid(True, linestyle='--', alpha=0.6)
    
    sns.lineplot(data=df, x='Epoch', y='R2', color='#1971c2', linewidth=2.5, ax=axes[1])
    axes[1].set_title('$R^2$ over Epochs', fontsize=16, pad=15)
    axes[1].set_xlabel('Epoch', fontsize=14)
    axes[1].set_ylabel('$R^2$ Score', fontsize=14)
    axes[1].grid(True, linestyle='--', alpha=0.6)
    
    sns.despine()
    plt.tight_layout()
    
    save_path = './results/PRnet_Bio_En/Parsed_MSE_R2_Curve_Auto.png'
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300)
    print(f"✅ 图片已完美保存至: {save_path}")
    plt.show()
    
else:
    print("❌ 搜索完毕。很遗憾，原作者的代码可能并没有把这些验证结果保存成文件。")
    print("\n💡 【补救建议】：")
    print("这说明程序只把结果打印在了屏幕内存里（也就是所谓的 Standard Output），而随着刷屏，早期的记录已经被系统自动丢弃了。")
    print("不用沮丧！你现在手里已经有了刚才画的【Loss 下降图】（那个是最关键的），完全足够向老师证明模型收敛。")
    print("如果你以后需要训练新的模型，记得在终端运行命令时加上重定向符号，例如：")
    print("👉  python train.py > training_log.txt  👈")
    print("这样不管刷多少屏，所有的进度条和 MSE 都会被死死钉在文本文件里，再也不会丢了！")

🕵️ 正在全盘搜索包含 MSE 和 R2 的日志文件，请稍候...
❌ 搜索完毕。很遗憾，原作者的代码可能并没有把这些验证结果保存成文件。

💡 【补救建议】：
这说明程序只把结果打印在了屏幕内存里（也就是所谓的 Standard Output），而随着刷屏，早期的记录已经被系统自动丢弃了。
不用沮丧！你现在手里已经有了刚才画的【Loss 下降图】（那个是最关键的），完全足够向老师证明模型收敛。
如果你以后需要训练新的模型，记得在终端运行命令时加上重定向符号，例如：
👉  python train.py > training_log.txt  👈
这样不管刷多少屏，所有的进度条和 MSE 都会被死死钉在文本文件里，再也不会丢了！
